# Campania coastal forecast comparison

Compare current WRF conditions for the gulfs of Naples and Salerno and the Cilento coast, following the reference use case’s reproducible indicator-and-figure pattern.

This live demonstration uses public meteo@uniparthenope data. Values are generated
automatically and are intended for research, education, and general information—not for official
warnings, navigation, or safety-critical decisions. Times are UTC. Run and edit the cells in this
browser-based workspace; your changes stay in your browser unless you download the notebook.


In [ ]:
import json
import os
import sys
from urllib.parse import urlencode
from datetime import datetime

if sys.platform == "emscripten":
    import piplite
    await piplite.install("ipywidgets")

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, clear_output, display

API_BASE = "https://api.meteo.uniparthenope.it" if sys.platform == "emscripten" else os.environ.get(
    "METEO_API_BASE_URL", "https://api.meteo.uniparthenope.it"
).rstrip("/")
BRAND = "#1f4e79"
ACCENT = "#2a91eb"

def api_json(path, params=None):
    url = f"{API_BASE}/{path.lstrip('/')}"
    if params:
        url = f"{url}?{urlencode(params)}"
    if sys.platform == "emscripten":
        from pyodide.http import open_url
        return json.load(open_url(url))
    import requests
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return response.json()

def forecast_frame(place):
    payload = api_json(f"products/wrf5/timeseries/{place}")
    if payload.get("result") != "ok":
        raise ValueError(f"No WRF forecast is available for {place}")
    frame = pd.DataFrame(payload["timeseries"])
    frame["time"] = pd.to_datetime(frame["dateTime"], format="%Y%m%dZ%H%M", utc=True)
    return frame.set_index("time")

def records_frame(payload):
    """Normalize common API list, GeoJSON, and envelope responses."""
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict) and isinstance(payload.get("features"), list):
        records = []
        for feature in payload["features"]:
            record = dict(feature.get("properties", {}))
            coordinates = feature.get("geometry", {}).get("coordinates", [])
            if len(coordinates) >= 2:
                record.update(longitude=coordinates[0], latitude=coordinates[1])
            records.append(record)
    elif isinstance(payload, dict):
        records = next((payload[key] for key in ("weatherreports", "reports", "data", "results", "items")
                        if isinstance(payload.get(key), list)), [payload])
    else:
        records = []
    return pd.json_normalize(records)

def numeric_snapshot(payload):
    """Return scalar numeric values without assuming a product-specific schema."""
    frame = records_frame(payload)
    if frame.empty:
        return frame
    numeric = frame.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    return numeric

PLACES = {"Gulf of Naples": "ca001", "Gulf of Salerno": "ca002", "Cilento": "ca003"}


In [ ]:
refresh = widgets.Button(description="Compare areas", button_style="primary", icon="refresh")
output = widgets.Output()

def render(_=None):
    with output:
        clear_output(wait=True)
        try:
            rows = []
            for label, identifier in PLACES.items():
                data = api_json(f"products/wrf5/forecast/{identifier}")
                rows.append({
                    "area": label,
                    "time": pd.to_datetime(data["dateTime"], format="%Y%m%dZ%H%M", utc=True),
                    "temperature_C": data["t2c"],
                    "humidity_pct": data["rh2"],
                    "wind_ms": data["ws10"],
                    "rain_mm": data["crh"],
                    "pressure_hPa": data["slp"],
                })
            frame = pd.DataFrame(rows).set_index("area")
            display(frame)
            axes = frame[["temperature_C", "humidity_pct", "wind_ms", "rain_mm"]].plot.bar(
                subplots=True, layout=(2, 2), figsize=(10, 7), legend=False, color=[BRAND, ACCENT, "#181c63"]
            )
            plt.suptitle("Current WRF indicators across Campania coastal areas")
            plt.tight_layout()
            plt.show()
        except Exception as exc:
            display(HTML(f"<b>Comparison failed:</b> {exc}"))

refresh.on_click(render)
display(widgets.VBox([refresh, output]))
render()
